# Model Comparison: Classification & Regression (Robust V7 Edition)

In [11]:
import os
import warnings
import numpy as np
import pandas as pd
import time

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.utils import resample

# Metrics
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    mean_absolute_error, r2_score
)

# Models
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier, MLPRegressor

warnings.filterwarnings("ignore")

In [12]:
# -----------------------------
# 1) HELPER FUNCTIONS
# -----------------------------
def _lower_cols(df):
    df = df.copy()
    df.columns = [c.strip().replace(" ", "_").lower() for c in df.columns]
    return df

def safe_div(a, b, default=0.0):
    try: return float(a)/float(b) if b and not pd.isna(b) else default
    except: return default

def _parse_date(x):
    try: return pd.to_datetime(x, errors="coerce")
    except: return np.nan

def extract_label(summary):
    text = str(summary).lower()
    if any(k in text for k in ["ui", "react", "frontend", "css", "html"]): return "frontend"
    if any(k in text for k in ["api", "backend", "server", "database"]): return "backend"
    if any(k in text for k in ["deploy", "docker", "pipeline", "ci", "cd"]): return "devops"
    if any(k in text for k in ["test", "qa", "bug"]): return "testing"
    return "general"

In [13]:
# -----------------------------
# LOAD DATA (V7 ROBUST LOGIC)
# -----------------------------
PROJECTS = {
    "Aurora": {
        "issues":  r"PM_Kaggle_dataset\AgileScrumSprintVelocityDataSet\Agile Scrum Dataset\Finalized Datasets for Aurora Project\Aurora Issues 554.csv",
        "sprints": r"PM_Kaggle_dataset\AgileScrumSprintVelocityDataSet\Agile Scrum Dataset\Finalized Datasets for Aurora Project\Aurora Sprints 41.csv",
    }
}

def load_and_aggregate_v7():
    combined = []
    for pname, p in PROJECTS.items():
        iss = _lower_cols(pd.read_csv(p["issues"], on_bad_lines="skip", engine="python"))
        spr = _lower_cols(pd.read_csv(p["sprints"], on_bad_lines="skip", engine="python"))
        iss.rename(columns={"priority": "priorityid", "storypoints": "storypoint"}, inplace=True)
        
        # Calculate Duration
        if "sprintstartdate" in spr.columns and "sprintcompletedate" in spr.columns:
             spr["start"] = pd.to_datetime(spr["sprintstartdate"], format="%d-%m-%Y %H:%M", errors="coerce")
             spr["end"]   = pd.to_datetime(spr["sprintcompletedate"], format="%d-%m-%Y %H:%M", errors="coerce")
             spr["actual_duration"] = (spr["end"] - spr["start"]).dt.days
             if "sprintlength" in spr.columns:
                 spr["actual_duration"] = spr["actual_duration"].fillna(spr["sprintlength"])
        elif "sprintlength" in spr.columns:
             spr["actual_duration"] = spr["sprintlength"]

        valid_sprints = spr["sprintid"].unique()
        iss = iss[iss["sprint"].isin(valid_sprints)]

        # Merging with Summary for Classification Tasks later
        # Note: Ideally we load summary file too, but for robust regression we stick to aggregations
        # For classification, we need row-level. We will do a hybrid return.
        
        agg = iss.groupby("sprint").agg({
            "storypoint": "sum",
            "priorityid": "mean",
            "key": "count"
        }).reset_index()
        
        full = agg.merge(spr[["sprintid", "actual_duration"]], left_on="sprint", right_on="sprintid")
        full["points_per_issue"] = full["storypoint"] / full["key"]
        full["complexity"] = full["storypoint"] * (4 - full["priorityid"].fillna(2))
        full["project"] = pname
        combined.append(full)
        
    return pd.concat(combined, ignore_index=True)

def load_raw_for_classification():
    # Re-using old logic just for classification to keep it working if needed
    # But user specifically asked about Regression Benchmark
    pass

df_v7_reg = load_and_aggregate_v7()
df_v7_reg = df_v7_reg.dropna(subset=["actual_duration"])

# Strict Filtering (V7 Logic)
df_reg_clean = df_v7_reg[(df_v7_reg["actual_duration"] >= 7) & (df_v7_reg["actual_duration"] <= 35)].copy()
print(f"V7 Clean Data (Regression): {len(df_reg_clean)} rows (7-35 days)")

V7 Clean Data (Regression): 38 rows (7-35 days)


## Task 2: Regression (Deadline Prediction)
Target: `actual_duration` (Real Days)  
Metric: Mean Absolute Error (MAE)

In [14]:
# 1. Select V7 Features and Target
target = "actual_duration"
features = ["storypoint", "priorityid", "key", "points_per_issue", "complexity"]

X_reg = df_reg_clean[features].fillna(0)
y_reg = df_reg_clean[target]

# 2. Split
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# 3. Preprocessor (Just Scaling)
preprocessor_reg = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scl", RobustScaler())
])

In [15]:
results_reg = []
print("Running Regression Benchmarks...")

models_reg = [
    ("Gradient Boosting", GradientBoostingRegressor(random_state=42)),
    ("XGBoost", XGBRegressor(random_state=42, n_jobs=-1)),
    ("Random Forest", RandomForestRegressor(random_state=42, n_jobs=-1)),
    ("SVM (SVR)", SVR(C=1.0, epsilon=0.2)),
    ("Neural Network", MLPRegressor(hidden_layer_sizes=(50, 50), max_iter=500, random_state=42))
]

for name, model in models_reg:
    start = time.time()
    
    # Pipeline
    pipe = Pipeline([
        ("preprocessor", preprocessor_reg),
        ("model", model)
    ])
    
    # Train
    pipe.fit(Xr_train, yr_train)
    
    # Evaluate
    y_pred = pipe.predict(Xr_test)
    mae = mean_absolute_error(yr_test, y_pred)
    r2  = r2_score(yr_test, y_pred)
    
    elapsed = time.time() - start
    print(f"  -> {name}: MAE={mae:.3f} Days, R2={r2:.3f} ({elapsed:.2f}s)")
    results_reg.append({"Model": name, "MAE": mae, "R2 Score": r2, "Time (s)": elapsed})

df_res_reg = pd.DataFrame(results_reg).sort_values("MAE")
display(df_res_reg)

Running Regression Benchmarks...
  -> Gradient Boosting: MAE=1.152 Days, R2=0.366 (0.04s)
  -> XGBoost: MAE=0.708 Days, R2=0.731 (0.05s)
  -> Random Forest: MAE=1.224 Days, R2=0.276 (0.17s)
  -> SVM (SVR): MAE=1.258 Days, R2=0.154 (0.01s)
  -> Neural Network: MAE=1.636 Days, R2=-0.226 (0.15s)


,Model,MAE,R2 Score,Time (s)
1,XGBoost,0.708327,0.730667,0.047267
0,Gradient Boosting,1.151567,0.366315,0.037921
2,Random Forest,1.223750,0.275605,0.170865
3,SVM (SVR),1.258250,0.153787,0.010205
4,Neural Network,1.635733,-0.225936,0.151146
